# 05 Reporting

Summarize HRV metrics and calibration results. Generate plots.

In [16]:
from pathlib import Path

calib_path = Path("artifacts/models/calibration_summary.csv")
print("Exists:", calib_path.exists())
print("Size (bytes):", calib_path.stat().st_size if calib_path.exists() else "File missing")


Exists: True
Size (bytes): 2


In [15]:
from pathlib import Path

models_dir = Path("artifacts") / "models"
metrics_dir = Path("artifacts") / "metrics"

metrics_df = pd.read_csv(metrics_dir / "hrv_metrics.csv")
calib_df = pd.read_csv(models_dir / "calibration_summary.csv")


EmptyDataError: No columns to parse from file

In [7]:
# HRV Summary
display(metrics_df.describe(include='all'))

,record,dataset,BVP_HRV_MeanNN_mean,BVP_HRV_SDNN_mean,BVP_HRV_SDANN1_mean,BVP_HRV_SDNNI1_mean,BVP_HRV_SDANN2_mean,BVP_HRV_SDNNI2_mean,BVP_HRV_SDANN5_mean,BVP_HRV_SDNNI5_mean,...,ECG_HRV_SampEn_sd,ECG_HRV_ShanEn_sd,ECG_HRV_FuzzyEn_sd,ECG_HRV_MSEn_sd,ECG_HRV_CMSEn_sd,ECG_HRV_RCMSEn_sd,ECG_HRV_CD_sd,ECG_HRV_HFD_sd,ECG_HRV_KFD_sd,ECG_HRV_LZC_sd
count,15,15,15.000000,15.000000,15.000000,15.000000,0.0,0.0,0.0,0.0,...,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
unique,15,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,S10,ppg_dalia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,1721.398334,1011.361868,279.928131,993.986915,NaN,NaN,NaN,NaN,...,0.451168,0.401820,0.281430,0.364587,0.254737,0.469046,0.300375,0.119041,0.570201,0.225584
std,NaN,NaN,219.600786,220.408842,136.241883,219.107766,NaN,NaN,NaN,NaN,...,0.078240,0.095747,0.061141,0.059654,0.035618,0.081847,0.071445,0.050486,0.197204,0.041714
min,NaN,NaN,1456.831737,490.739665,152.881837,462.311482,NaN,NaN,NaN,NaN,...,0.293780,0.285529,0.186017,0.246159,0.205891,0.329505,0.202361,0.069179,0.317551,0.171468
25%,NaN,NaN,1578.337727,919.119884,223.704721,905.682172,NaN,NaN,NaN,NaN,...,0.397874,0.325612,0.240990,0.325154,0.227736,0.410502,0.242719,0.078303,0.426442,0.200699
50%,NaN,NaN,1646.417330,1058.085388,260.447571,1037.483052,NaN,NaN,NaN,NaN,...,0.457736,0.390535,0.281839,0.377645,0.250601,0.475016,0.311943,0.125680,0.548704,0.231261
75%,NaN,NaN,1795.446675,1101.889155,282.609436,1095.973805,NaN,NaN,NaN,NaN,...,0.516878,0.463702,0.315365,0.394191,0.273872,0.525683,0.336917,0.135094,0.647048,0.237067


In [8]:
# Calibration Summary
display(calib_df)

,feature,coef,intercept,mae,rmse,r2
0,HRV_MeanNN_mean,0.076520,592.294033,64.391062,88.493856,0.032558
1,HRV_SDNN_mean,0.207663,-115.167591,31.459194,48.033231,0.458722
2,HRV_SDANN1_mean,0.081189,27.250396,6.718951,8.490106,0.613039
3,HRV_SDNNI1_mean,0.177849,-102.232020,33.832534,54.432631,0.323565
4,HRV_RMSSD_mean,0.164348,-170.991941,50.805582,71.442576,0.390164
...,...,...,...,...,...,...
165,HRV_RCMSEn_sd,-0.306571,0.610064,0.062683,0.075940,0.077648
166,HRV_CD_sd,0.131811,0.246834,0.057493,0.068833,0.005497
167,HRV_HFD_sd,1.839146,0.017114,0.032146,0.045518,0.129048
168,HRV_KFD_sd,0.017256,0.513879,0.145261,0.178251,0.124625


In [9]:
# ===============================
# SECTION: Calibration Plots
# ===============================
report_dir = Path("artifacts/reports")
report_dir.mkdir(parents=True, exist_ok=True)

# Load calibration summary
calib_df = pd.read_csv(models_dir / "calibration_summary.csv")

if calib_df.empty:
    print("No calibration results to plot.")
else:
    for feat in calib_df["feature"].tolist():
        xcol = f"BVP_{feat}"
        ycol = f"ECG_{feat}"

        if xcol not in metrics_df.columns or ycol not in metrics_df.columns:
            log(f"Skipped plotting {feat}: missing columns", "WARN")
            continue

        x = metrics_df[xcol]
        y = metrics_df[ycol]

        if pd.isna(x).any() or pd.isna(y).any():
            log(f"Skipped plotting {feat}: NaN values present", "WARN")
            continue

        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=x, y=y, alpha=0.6)
        sns.regplot(x=x, y=y, scatter=False, color="red")
        plt.xlabel(xcol)
        plt.ylabel(ycol)
        plt.title(f"Calibration: {feat}")
        out_file = report_dir / f"calibration_{feat}.png"
        plt.savefig(out_file, dpi=150, bbox_inches="tight")
        plt.close()
        log(f"Saved calibration plot for {feat} → {out_file}", "OK")

print("Calibration plots saved to:", report_dir)


[2025-09-25 21:01:21] [OK] Saved calibration plot for HRV_MeanNN_mean → artifacts\reports\calibration_HRV_MeanNN_mean.png
[2025-09-25 21:01:21] [OK] Saved calibration plot for HRV_SDNN_mean → artifacts\reports\calibration_HRV_SDNN_mean.png
[2025-09-25 21:01:21] [OK] Saved calibration plot for HRV_SDANN1_mean → artifacts\reports\calibration_HRV_SDANN1_mean.png
[2025-09-25 21:01:21] [OK] Saved calibration plot for HRV_SDNNI1_mean → artifacts\reports\calibration_HRV_SDNNI1_mean.png
[2025-09-25 21:01:21] [OK] Saved calibration plot for HRV_RMSSD_mean → artifacts\reports\calibration_HRV_RMSSD_mean.png
[2025-09-25 21:01:22] [OK] Saved calibration plot for HRV_SDSD_mean → artifacts\reports\calibration_HRV_SDSD_mean.png
[2025-09-25 21:01:22] [OK] Saved calibration plot for HRV_CVNN_mean → artifacts\reports\calibration_HRV_CVNN_mean.png
[2025-09-25 21:01:22] [OK] Saved calibration plot for HRV_CVSD_mean → artifacts\reports\calibration_HRV_CVSD_mean.png
[2025-09-25 21:01:22] [OK] Saved calibrati

In [11]:
# ===============================
# SECTION: Self-contained HTML Report
# ===============================
import base64

report_dir = Path("artifacts/reports")
report_dir.mkdir(parents=True, exist_ok=True)

summary_md = report_dir / "summary_report.md"
summary_html = report_dir / "summary_report.html"

# --- HRV Summary Stats ---
hrv_stats = metrics_df.describe(include="all").to_markdown()

# --- Calibration Summary Table ---
calib_table = calib_df.to_markdown(index=False)

# --- Embed calibration plots as base64 ---
plot_files = sorted(report_dir.glob("calibration_*.png"))
plot_html_blocks = []
for p in plot_files:
    with open(p, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")
    plot_html_blocks.append(f"<h3>{p.stem}</h3><img src='data:image/png;base64,{img_b64}' width='500'>")

# --- Assemble Markdown (for .md file) ---
plot_md_links = "\n".join([f"![{p.stem}]({p.name})" for p in plot_files])
md_content = f"""
# HRV-GenAI Project Report

Generated on: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

---

## HRV Metrics Summary

{hrv_stats}

---

## Calibration Summary

{calib_table}

---

## Calibration Plots

{plot_md_links}
"""
with open(summary_md, "w", encoding="utf-8") as f:
    f.write(md_content)

# --- Assemble HTML ---
html_content = f"""
<html>
<head>
  <title>HRV-GenAI Project Report</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 20px; }}
    table {{ border-collapse: collapse; margin: 10px 0; }}
    th, td {{ border: 1px solid #ccc; padding: 6px 10px; }}
    h1, h2, h3 {{ color: #333; }}
  </style>
</head>
<body>
  <h1>HRV-GenAI Project Report</h1>
  <p><em>Generated on {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</em></p>

  <h2>HRV Metrics Summary</h2>
  {metrics_df.describe(include="all").to_html()}

  <h2>Calibration Summary</h2>
  {calib_df.to_html(index=False)}

  <h2>Calibration Plots</h2>
  {''.join(plot_html_blocks)}

</body>
</html>
"""
with open(summary_html, "w", encoding="utf-8") as f:
    f.write(html_content)

log(f"Saved Markdown report → {summary_md}", "DONE")
log(f"Saved self-contained HTML report → {summary_html}", "DONE")


[2025-09-25 21:17:21] [DONE] Saved Markdown report → artifacts\reports\summary_report.md
[2025-09-25 21:17:21] [DONE] Saved self-contained HTML report → artifacts\reports\summary_report.html
